In [26]:
city_coords = {
    "Berlin": (52.5200, 13.4050),
    "Hamburg": (53.5511, 9.9937),
    "Munich": (48.1351, 11.5820),
    "Frankfurt": (50.1109, 8.6821),
    "Stuttgart": (48.7758, 9.1829),
    "Dresden": (51.0504, 13.7373),
    "Düsseldorf": (51.2277, 6.7735),
    "Erfurt": (50.9848, 11.0299),
    "Hanover": (52.3759, 9.7320),
    "Magdeburg": (52.1205, 11.6276),
    "Mainz": (49.9929, 8.2473),
    "Rostock": (54.0924, 12.0991)
}

In [27]:
import requests
import pandas as pd

def get_past7_pollen(city, lat, lon):
    """
    Fetch past 7 days grass pollen from Open-Meteo.
    """
    url = (
        "https://air-quality-api.open-meteo.com/v1/air-quality"
        f"?latitude={lat}&longitude={lon}"
        "&hourly=grass_pollen"
        "&past_days=7"
        "&forecast_days=0"
    )

    response = requests.get(url).json()

    times = response["hourly"]["time"]
    grass = response["hourly"]["grass_pollen"]

    df = pd.DataFrame({
        "city": city,
        "datetime": times,
        "grass_pollen": grass
    })

    # Convert to daily average
    df["date"] = pd.to_datetime(df["datetime"]).dt.date
    df_daily = df.groupby(["city", "date"], as_index=False)["grass_pollen"].mean()

    return df_daily


In [28]:
all_data = []

for city, (lat, lon) in city_coords.items():
    print(f"Fetching past 7 days pollen for {city}...")
    df_city = get_past7_pollen(city, lat, lon)
    all_data.append(df_city)

df_pollen_past7 = pd.concat(all_data, ignore_index=True)

df_pollen_past7


Fetching past 7 days pollen for Berlin...
Fetching past 7 days pollen for Hamburg...
Fetching past 7 days pollen for Munich...
Fetching past 7 days pollen for Frankfurt...
Fetching past 7 days pollen for Stuttgart...
Fetching past 7 days pollen for Dresden...
Fetching past 7 days pollen for Düsseldorf...
Fetching past 7 days pollen for Erfurt...
Fetching past 7 days pollen for Hanover...
Fetching past 7 days pollen for Magdeburg...
Fetching past 7 days pollen for Mainz...
Fetching past 7 days pollen for Rostock...


,city,date,grass_pollen
0,Berlin,2026-05-11,0.962500
1,Berlin,2026-05-12,1.108333
2,Berlin,2026-05-13,0.850000
3,Berlin,2026-05-14,1.254167
4,Berlin,2026-05-15,2.112500
...,...,...,...
79,Rostock,2026-05-13,1.245833
80,Rostock,2026-05-14,1.454167
81,Rostock,2026-05-15,1.941667
82,Rostock,2026-05-16,2.095833


In [29]:
df_pollen_past7["city"].unique()


array(['Berlin', 'Hamburg', 'Munich', 'Frankfurt', 'Stuttgart', 'Dresden',
       'Düsseldorf', 'Erfurt', 'Hanover', 'Magdeburg', 'Mainz', 'Rostock'],
      dtype=object)

✅ 1. Weather Function (Past 7 Days)

In [30]:
import requests
import pandas as pd

def get_past7_weather(city, lat, lon):
    """
    Fetch past 7 days weather from Open-Meteo.
    """
    url = (
        "https://api.open-meteo.com/v1/forecast"
        f"?latitude={lat}&longitude={lon}"
        "&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation"
        "&past_days=7"
        "&forecast_days=0"
        "&timezone=auto"
    )

    response = requests.get(url).json()

    times = response["hourly"]["time"]
    temp = response["hourly"]["temperature_2m"]
    humidity = response["hourly"]["relative_humidity_2m"]
    wind = response["hourly"]["wind_speed_10m"]
    rain = response["hourly"]["precipitation"]

    df = pd.DataFrame({
        "city": city,
        "datetime": times,
        "temperature": temp,
        "humidity": humidity,
        "wind_speed": wind,
        "precipitation": rain
    })

    # Convert to daily values
    df["date"] = pd.to_datetime(df["datetime"]).dt.date

    df_daily = df.groupby(["city", "date"], as_index=False).agg({
        "temperature": "mean",
        "humidity": "mean",
        "wind_speed": "mean",
        "precipitation": "sum"   # rain = daily total
    })

    return df_daily


In [31]:
#✅ 2. Loop Through All 12 Cities
weather_data = []

for city, (lat, lon) in city_coords.items():
    print(f"Fetching past 7 days weather for {city}...")
    df_city_weather = get_past7_weather(city, lat, lon)
    weather_data.append(df_city_weather)

df_weather_past7 = pd.concat(weather_data, ignore_index=True)

df_weather_past7


Fetching past 7 days weather for Berlin...
Fetching past 7 days weather for Hamburg...
Fetching past 7 days weather for Munich...
Fetching past 7 days weather for Frankfurt...
Fetching past 7 days weather for Stuttgart...
Fetching past 7 days weather for Dresden...
Fetching past 7 days weather for Düsseldorf...
Fetching past 7 days weather for Erfurt...
Fetching past 7 days weather for Hanover...
Fetching past 7 days weather for Magdeburg...
Fetching past 7 days weather for Mainz...
Fetching past 7 days weather for Rostock...


,city,date,temperature,humidity,wind_speed,precipitation
0,Berlin,2026-05-11,13.108333,70.541667,12.375000,1.5
1,Berlin,2026-05-12,8.537500,68.458333,11.445833,8.0
2,Berlin,2026-05-13,9.412500,73.500000,11.191667,2.3
3,Berlin,2026-05-14,10.191667,73.666667,9.637500,4.7
4,Berlin,2026-05-15,11.195833,66.125000,5.620833,0.0
...,...,...,...,...,...,...
79,Rostock,2026-05-13,8.775000,77.250000,12.483333,1.4
80,Rostock,2026-05-14,9.195833,78.208333,8.537500,5.1
81,Rostock,2026-05-15,10.054167,70.666667,7.104167,0.1
82,Rostock,2026-05-16,9.483333,74.333333,12.870833,2.9


In [32]:
past_7days_df = df_pollen_past7.merge(df_weather_past7, on=["date", "city"], how="inner")
past_7days_df.to_csv("combined_7days.csv", index=False)


In [33]:
past_7days_df

,city,date,grass_pollen,temperature,humidity,wind_speed,precipitation
0,Berlin,2026-05-11,0.962500,13.108333,70.541667,12.375000,1.5
1,Berlin,2026-05-12,1.108333,8.537500,68.458333,11.445833,8.0
2,Berlin,2026-05-13,0.850000,9.412500,73.500000,11.191667,2.3
3,Berlin,2026-05-14,1.254167,10.191667,73.666667,9.637500,4.7
4,Berlin,2026-05-15,2.112500,11.195833,66.125000,5.620833,0.0
...,...,...,...,...,...,...,...
79,Rostock,2026-05-13,1.245833,8.775000,77.250000,12.483333,1.4
80,Rostock,2026-05-14,1.454167,9.195833,78.208333,8.537500,5.1
81,Rostock,2026-05-15,1.941667,10.054167,70.666667,7.104167,0.1
82,Rostock,2026-05-16,2.095833,9.483333,74.333333,12.870833,2.9
